<a href="https://colab.research.google.com/github/mjspapa/Utility-Scale-Solar-Performance-and-Diagnostic-Pipeline/blob/main/Solar_Plant_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the Data
print("Loading datasets...")
gen_df = pd.read_csv('Plant_1_Generation_Data.csv')
weather_df = pd.read_csv('Plant_1_Weather_Sensor_Data.csv')

# 2. Standardize Timestamps (The Data Engineering Fix)
# Generation data uses DD-MM-YYYY, Weather data uses YYYY-MM-DD
print("Aligning timestamps...")
gen_df['DATE_TIME'] = pd.to_datetime(gen_df['DATE_TIME'], format='%d-%m-%Y %H:%M')
weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')

# 3. Clean and Merge
# Rename weather sensor key to avoid confusion with inverter source key
weather_df = weather_df.rename(columns={'SOURCE_KEY': 'WEATHER_SENSOR_KEY'})
weather_df = weather_df.drop(columns=['PLANT_ID']) # Drop duplicate ID column

# Left join: Attach weather conditions to every inverter's timestamp
df = pd.merge(gen_df, weather_df, on='DATE_TIME', how='left')

# Forward-fill any tiny gaps in the weather sensor data
df['IRRADIATION'] = df['IRRADIATION'].ffill()
df['AMBIENT_TEMPERATURE'] = df['AMBIENT_TEMPERATURE'].ffill()
df['MODULE_TEMPERATURE'] = df['MODULE_TEMPERATURE'].ffill()

# 4. Feature Engineering
print("Calculating performance metrics...")

# A. Inverter Conversion Efficiency
# Note: In this specific Kaggle dataset, DC_POWER is scaled by a factor of 10 relative to AC_POWER.
df['INVERTER_EFFICIENCY'] = np.where(
    df['DC_POWER'] > 0,
    df['AC_POWER'] / (df['DC_POWER'] / 10),
    0
)

# B. Soiling Index (Normalized DC Output)
# Only calculate this when the sun is actually shining (Irradiation > 0.1) to remove night/cloud noise.
df['NORMALIZED_DC_EFFICIENCY'] = np.where(
    df['IRRADIATION'] > 0.1,
    df['DC_POWER'] / df['IRRADIATION'],
    0
)

# C. Thermal Anomalies
# Calculate the exact heat buildup on the panels compared to the surrounding air.
df['TEMP_DELTA'] = df['MODULE_TEMPERATURE'] - df['AMBIENT_TEMPERATURE']

# D. Peak Sun Hours Flag (For filtering in Tableau)
df['HOUR'] = df['DATE_TIME'].dt.hour
df['IS_PEAK_SUN'] = np.where((df['HOUR'] >= 10) & (df['HOUR'] <= 14), 1, 0)

# 5. Export for Tableau
output_filename = 'Solar_Performance_Tableau_Ready.csv'
df.to_csv(output_filename, index=False)

print(f"\nPipeline complete! Successfully processed {len(df)} records.")
print(f"File saved as: {output_filename}")
display(df.head())

Loading datasets...
Aligning timestamps...
Calculating performance metrics...

Pipeline complete! Successfully processed 68778 records.
File saved as: Solar_Performance_Tableau_Ready.csv


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,WEATHER_SENSOR_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION,INVERTER_EFFICIENCY,NORMALIZED_DC_EFFICIENCY,TEMP_DELTA,HOUR,IS_PEAK_SUN
0,2020-05-15,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0,0.0,0.0,-2.326809,0,0
1,2020-05-15,4135001,1IF53ai7Xc0U56Y,0.0,0.0,0.0,6183645.0,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0,0.0,0.0,-2.326809,0,0
2,2020-05-15,4135001,3PZuoBAID5Wc2HD,0.0,0.0,0.0,6987759.0,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0,0.0,0.0,-2.326809,0,0
3,2020-05-15,4135001,7JYdWkrLSPkdwr4,0.0,0.0,0.0,7602960.0,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0,0.0,0.0,-2.326809,0,0
4,2020-05-15,4135001,McdE0feGgRqW7Ca,0.0,0.0,0.0,7158964.0,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0,0.0,0.0,-2.326809,0,0
